# Visualización de resultados

## Equipo 74  
* Diana Jimena López Nájera
* Marcelo Ismael López Verdugo
* Salma Alejandra Macías Rosas
* Dario Maximiliano Mendoza Orozco

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, when, isnan, lit, desc, hour, dayofmonth, dayofweek
from pyspark.sql.functions import split, regexp_extract
from pyspark.sql.functions import collect_set, countDistinct, collect_list
from pyspark.sql.types import DoubleType, FloatType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import DecisionTreeRegressor, GBTRegressor
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator




spark = SparkSession.builder \
    .appName("Analisis Ecommerce Octubre") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

spark

## Directorio del dataset

In [2]:
root=".."
file_path = root + "/datasets/2019-Oct.csv"

df = spark.read.option("header", True) \
               .option("inferSchema", True) \
               .csv(file_path)

df.show(5, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-09-30 17:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-09-30 17:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-09-30 17:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-09-30 17:0

In [3]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



# Depuración y preparación

## Análisis de cardinalidad
Es útil conocer cuantos valores únicos existen por columna para identificar potenciales particionamientos

In [4]:
df_clean = df.dropna()
df_clean = df_clean.withColumn("weekhour", dayofweek(col("event_time")) + hour(col("event_time")) / 24)

non_numeric_columns = [name for name, dtype in df_clean.dtypes if dtype == "string"]
for col_name in non_numeric_columns:
    distinct_count = df_clean.select(col_name).distinct().count()
    print(f"{col_name}: dtype=string, distinct={distinct_count}")

    top_values = [row[col_name] for row in df_clean.groupBy(col_name)
                                              .count()
                                              .orderBy(desc("count"))
                                              .limit(10)
                                              .select(col_name)
                                              .collect()]

    grouped_col = f"{col_name}_grouped"
    df_clean = df_clean.withColumn(
        grouped_col,
        when(col(col_name).isin(top_values), col(col_name)).otherwise(lit("other"))
    )

print("Columnas agrupadas creadas:", [f"{c}_grouped" for c in non_numeric_columns])

df_clean.describe().show()

event_type: dtype=string, distinct=3
category_code: dtype=string, distinct=126
brand: dtype=string, distinct=1731
user_session: dtype=string, distinct=6419693
Columnas agrupadas creadas: ['event_type_grouped', 'category_code_grouped', 'brand_grouped', 'user_session_grouped']


In [ ]:
#Valores únicos por variable
columns_to_check = ["product_id", "category_id", "category_code", "brand", "user_id", "user_session"]

for c in columns_to_check:
    print(f"{c}: {df.select(c).distinct().count():,} valores únicos")

product_id: 166,794 valores únicos
category_id: 624 valores únicos
category_code: 127 valores únicos
brand: 3,446 valores únicos
user_id: 3,022,290 valores únicos
user_session: 9,244,422 valores únicos


In [ ]:
## Clustering de clientes por usuario
# Agrupamos por user_id y calculamos métricas de interacción y precio.
user_features = (
    df_clean
    .groupBy("user_id")
    .agg(
        count("*").alias("interactions"),
        sum("price").alias("total_spend"),
        avg("price").alias("avg_price"),
        countDistinct("brand").alias("brand_diversity"),
        collect_list("brand").alias("brand_list")
    )
)

user_features.show(10, truncate=False)

# Vectorizamos las marcas con sparse features usando CountVectorizer.
from pyspark.ml.feature import CountVectorizer, StandardScaler
from pyspark.ml.clustering import KMeans

cv = CountVectorizer(inputCol="brand_list", outputCol="brand_vec", vocabSize=100, minDF=2.0)
assembler = VectorAssembler(
    inputCols=["interactions", "total_spend", "avg_price", "brand_diversity", "brand_vec"],
    outputCol="raw_features"
)
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)

pipeline = Pipeline(stages=[cv, assembler, scaler])
pipeline_model = pipeline.fit(user_features)
user_vector = pipeline_model.transform(user_features)

# Entrenamos KMeans y asignamos un segmento de cliente.
k = 4
kmeans = KMeans(k=k, featuresCol="features", predictionCol="customer_segment", seed=42)
kmeans_model = kmeans.fit(user_vector)
clustered_users = kmeans_model.transform(user_vector)

clustered_users.select(
    "user_id", "customer_segment", "interactions", "total_spend", "avg_price", "brand_diversity"
).show(20, truncate=False)

# Agregamos la categoría al dataset original para análisis posterior.
df_with_segment = df_clean.join(
    clustered_users.select("user_id", "customer_segment"),
    on="user_id",
    how="left"
)

print("Distribución de segmentos de cliente:")
df_with_segment.groupBy("customer_segment").count().orderBy("customer_segment").show()

# Opcional: ver centros de los clústeres.
print("Centros de cluster:")
for idx, center in enumerate(kmeans_model.clusterCenters()):
    print(f"Cluster {idx}: {center}")
